# 🎬 CineMatch — Cognizant Hackathon Showcase Notebook
### Track: Use Case #6 — OTT Content Recommendation and Engagement Agent

This interactive notebook demonstrates the core machine learning algorithms powering the **CineMatch / CineWatch AI** recommendation engine:
1. **Content-Based Filtering**: TF-IDF Feature Bag Vectorization + Cosine Similarity Matrix
2. **Collaborative Filtering**: Regularized SVD Matrix Factorization (50 Latent Factors)
3. **Hybrid Ensemble Fusion**: Weighted Score Blending ($0.6 \text{ Collab} + 0.4 \text{ Content}$)
4. **Explainable AI (XAI)**: Natural-language rationale and metadata tag attribution
5. **Behavioral Engagement Scoring**: Multi-factor subscriber engagement index ($E_u \in [0, 100]$) and churn retention cohorts

In [1]:
# 1. Environment Setup & Data Ingestion
import sys
import os
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('.'))
from data_loader import DataLoader
from content_engine import ContentEngine
from collaborative_engine import CollaborativeEngine
from hybrid_engine import HybridEngine
from explainer import Explainer
from engagement_scorer import EngagementScorer

print('Initializing CineMatch DataLoader...')
loader = DataLoader()
loader.load_movies()
loader.load_ratings(sample_size=150000)

print(f'\n[Dataset Metrics]')
print(f'Total Movies:  {len(loader.movies_df):,}')
print(f'Total Ratings: {len(loader.ratings_df):,}')

Initializing CineMatch DataLoader...
Loading movies from c:\CineMatch\DATASET\movies.csv...
Loaded 4760 movies successfully.
Loading ratings from c:\CineMatch\DATASET\ratings.csv...
Sampling 150,000 ratings for faster local processing...
Loaded 150,000 ratings from 7,007 users.

[Dataset Metrics]
Total Movies:  4,760
Total Ratings: 150,000


In [2]:
# 2. Content-Based Engine (TF-IDF + Cosine Similarity)
content_engine = ContentEngine(loader)
content_engine.fit()

# Query similar titles for 'Star Wars' (Movie ID: 2)
target_id = 2
recs = content_engine.get_content_recommendations(target_id, top_n=5)
print(f'Top Content Similarities for {loader.movie_id_to_title.get(target_id)}:')
for mid, score in recs:
    print(f'  - {loader.movie_id_to_title.get(mid)} | Similarity: {score:.4f}')

Fitting TF-IDF Vectorizer on movie feature bags...
TF-IDF Matrix shape: (4760, 10000)
Computing Cosine Similarity Matrix (4,760 x 4,760)...
Content Engine fitted successfully!
Top Content Similarities for Star Wars:
  - The Empire Strikes Back | Similarity: 0.4075
  - Return of the Jedi | Similarity: 0.2912
  - Star Wars: Clone Wars: Volume 1 | Similarity: 0.2653
  - Star Wars: Episode I - The Phantom Menace | Similarity: 0.2523
  - Star Wars: Episode III - Revenge of the Sith | Similarity: 0.2392


In [3]:
# 3. Collaborative Filtering Engine (SVD Matrix Factorization)
collab_engine = CollaborativeEngine(loader)
collab_engine.fit()

# Evaluate with 3-Fold Cross-Validation
metrics = collab_engine.evaluate(cv=3)
print(f'\nSVD Cross-Validation Benchmark:')
print(f'  RMSE: {metrics["rmse"]:.4f} (Target <= 0.88)')
print(f'  MAE:  {metrics["mae"]:.4f} (Target <= 0.68)')

Building full trainset for Collaborative Filtering (SVD)...
Training SVD model (n_factors=50, epochs=20)...
Collaborative Engine (SVD) fitted successfully!
Running 3-Fold Cross-Validation for SVD...
Evaluation Results -> RMSE: 0.9043 | MAE: 0.6965

SVD Cross-Validation Benchmark:
  RMSE: 0.9043 (Target <= 0.88)
  MAE:  0.6965 (Target <= 0.68)


In [7]:
# 4. Hybrid Fusion & Personalized Watchlist
hybrid_engine = HybridEngine(loader, content_engine, collab_engine)
test_user = 40
watchlist = hybrid_engine.recommend(test_user, top_n=5)

print(f'Top 5 Hybrid Recommendations for User #{test_user}:')
for item in watchlist:
    print(f'  - {item["title"]} ({item["year"]}) | Pred: {item["predictedRating"]}★ | Match: {item["matchPercentage"]}% | HybridScore: {item["hybridScore"]:.3f}')

Top 5 Hybrid Recommendations for User #40:
  - Tales from the Crypt: Demon Knight (1995) | Pred: 4.44★ | Match: 83% | HybridScore: 0.570
  - The Hunting Party (2007) | Pred: 4.49★ | Match: 82% | HybridScore: 0.556
  - Marie Antoinette (2006) | Pred: 4.42★ | Match: 82% | HybridScore: 0.548
  - Elizabeth (1998) | Pred: 3.76★ | Match: 82% | HybridScore: 0.541
  - Happily N'Ever After (2006) | Pred: 4.36★ | Match: 81% | HybridScore: 0.538


In [8]:
# 5. Explainable AI (XAI) Engine
explainer = Explainer(loader, content_engine, collab_engine)
first_rec = watchlist[0]
xai_info = explainer.explain_recommendation(test_user, first_rec['movieId'], candidate_info=first_rec)

print(f'XAI Explanation for {first_rec["title"]}:')
print(f'  Text: "{xai_info["text"]}"')
print(f'  Feature Tags: {xai_info["tags"]}')
print(f'  Peer Agreement: {xai_info["peerAgreementPct"]}%')

XAI Explanation for Tales from the Crypt: Demon Knight:
  Text: "Recommended based on your high affinity for Pretty Woman. Strong Comedy alignment with 93% peer agreement."
  Feature Tags: ['Dir. Ernest R. Dickerson', 'Comedy Match', '93% Peer Match']
  Peer Agreement: 93%


In [ ]:
# 6. Behavioral Engagement & Churn Retention Engine
engagement_scorer = EngagementScorer(loader)
for uid in [2177, 1, 99999]:
    res = engagement_scorer.calculate_user_engagement(uid)
    print(f'User #{uid}: Score {res["score"]}/100 | Cohort: {res["cohort"]} | Churn Risk: {res["churnRisk"]}')

User #2177: Score 85/100 | Cohort: Power Viewer | Churn Risk: Minimal (<5%)
User #1: Score 44/100 | Cohort: Casual Viewer | Churn Risk: Moderate (35%)
User #99999: Score 15/100 | Cohort: Dormant / At-Risk | Churn Risk: High (80%)


: 